# 3.2 Network-level CM-FC decoupling in MCI

In [ ]:
from ana_utils import *
import warnings
warnings.filterwarnings("ignore")
import svgutils

In [ ]:
output_dir = f"{PROJ_HOME}/results/2_local_CFC"
os.makedirs(output_dir, exist_ok=True)
print("Output dir:", output_dir)

# Convert colors in PAL4 and CM_ARR_COLOR to hex format
colors_hex = [mcolors.to_hex(color) for color in PAL4]
cm_arr_hex = [mcolors.to_hex(color) for color in CM_ARR_COLOR]

# Print the assigned variables for verification
print("Colors (hex):", colors_hex)
print("CM Array Colors (hex):", cm_arr_hex)

In [ ]:
# Merge CFC data for five LMs
merged_df = pd.concat(
    [
        pd.read_csv(f"{output_dir}/_lr-local_cfc-{tract_lab}.csv").assign(Tractography=tract_lab)
        for tract_lab in TRACT_LABS
    ],
    ignore_index=True,
)

# Calculate merged_diff_df (INT - WMH for each pid and Group)
int_sub_df = merged_df[merged_df.Tractography == TRACT_LABS[1]].set_index(["pid", "Group"])
wmh_sub_df = merged_df[merged_df.Tractography == TRACT_LABS[2]].set_index(["pid", "Group"])

merged_diff_df = (int_sub_df[CM_LABS] - wmh_sub_df[CM_LABS]).reset_index()

## Table S5. Comparison of the network-level CM-FC coupling R2(All)

In [ ]:
comp_res_df = pd.DataFrame(
    []
)
posthoc_res_df = pd.DataFrame(
    []
)
for net_idx, net_lab in enumerate(NET_LABS.keys()):
    anova_df = merged_df[merged_df.net==net_lab]
    res = pg_anova(anova_df[anova_df.Tractography==TRACT_LABS[0]], col_name="rsq", factor="Group")
    anova_res = res[0].assign(
        Network=NET_LABS[net_lab],
        Tractography=TRACT_LABS[0],
    )
    posthoc_res_df = pd.concat(
        [posthoc_res_df, res[1].assign(
            net=NET_LABS[net_lab])],
        ignore_index=True
        )
         
    comp_res_df = pd.concat(
        [comp_res_df, anova_res],
        ignore_index=True
    )
    
comp_res_df["P_corrected"] = add_statistic_annotation(fdr_correction(comp_res_df["p-unc"].to_list()))
print(comp_res_df["P_corrected"].value_counts())
posthoc_res_df["P"] = add_statistic_annotation(posthoc_res_df["pval"].to_list())
comp_res_df.round(3).to_csv(f"{output_dir}/local_CFC_comparison-R2-All.csv")
posthoc_res_df.round(3).to_csv(f"{output_dir}/local_CFC_comparison-R2-All-posthoc.csv")

## Table S5. Comparison of the network-level CM-FC coupling R2 (INT and WMH)

In [ ]:
# Two-way ANOVA between int_subtbl and wmh_subtbl
int_wmh_df = pd.concat(
    [
        int_sub_df.reset_index().assign(tg_lab="int"),
        wmh_sub_df.reset_index().assign(tg_lab="wmh"),
    ],
    ignore_index=True,
)

In [ ]:
anova_res_df = pd.DataFrame(
    []
)
posthoc_res_df = pd.DataFrame(
    []
)
for net_idx, net_lab in enumerate(NET_LABS.keys()):
    anova_df = int_wmh_df[int_wmh_df.net==net_lab]
    anova_res, posthoc = pg_mix_anova(anova_df)
    # print(posthoc)
    anova_res=anova_res.assign(
        Network=NET_LABS[net_lab],
    )
    anova_res_df = pd.concat(
        [anova_res_df, anova_res],
        ignore_index=True
    )
    posthoc_res_df = pd.concat(
        [posthoc_res_df, posthoc.assign(
            Network=NET_LABS[net_lab])],
        ignore_index=True
    )
    
anova_res_df["P"] = add_statistic_annotation(anova_res_df["p-unc"].to_list())
anova_res_df["P_corrected"] = anova_res_df.groupby("Source")["p-unc"].transform(
    lambda p: add_statistic_annotation(fdr_correction(p.tolist()))
)
posthoc_res_df["P"] = add_statistic_annotation(posthoc_res_df["pval"].to_list())
anova_res_df.round(3).to_csv(f"{output_dir}/local_CFC_comparison-R2-INT_WMH.csv")
posthoc_res_df.round(3).to_csv(f"{output_dir}/local_CFC_comparison-R2-INT_WMH-posthoc.csv")
print("Mixed ANOVA Result:")

## Table S6. Comparison of the dominance for local CM-FC coupling (All)

In [ ]:
posthoc_res_df = pd.DataFrame([])
anova_res_df = pd.DataFrame([])
for cm_lab in CM_LABS:
    cm_anova_res_df = pd.DataFrame([])

    for net_idx, net_lab in enumerate(NET_LABS.keys()):
        anova_df = merged_df[merged_df.net==net_lab]
        res = pg_anova(anova_df[anova_df.Tractography==TRACT_LABS[0]], col_name=cm_lab, factor="Group")
        anova_res = res[0].assign(
            Network=NET_LABS[net_lab],
            Tractography="All",
            CM=cm_lab[:2].upper(),
        )
        posthoc_res_df = pd.concat(
            [posthoc_res_df, res[1].assign(
                Network=NET_LABS[net_lab],
                CM=cm_lab[:2].upper()
            )],
            ignore_index=True
            )
            
        cm_anova_res_df = pd.concat(
            [cm_anova_res_df, anova_res],
            ignore_index=True
        )
    
    cm_anova_res_df["P_corrected"] = add_statistic_annotation(fdr_correction(cm_anova_res_df["p-unc"].to_list()))
    anova_res_df = pd.concat([anova_res_df, cm_anova_res_df], ignore_index=True)

posthoc_res_df["P"] = add_statistic_annotation(posthoc_res_df["pval"].to_list())
anova_res_df.round(3).to_csv(f"{output_dir}/local_CFC_comparison-Dominance-All.csv")
posthoc_res_df.round(3).to_csv(f"{output_dir}/local_CFC_comparison-Dominance-All-posthoc.csv")

## Table S6. Comparison of the dominance for local CM-FC coupling (INT-WMH)

In [ ]:
posthoc_res_df = pd.DataFrame([])
anova_res_df = pd.DataFrame([])
for cm_lab in CM_LABS:
    cm_anova_res_df = pd.DataFrame([])

    for net_idx, net_lab in enumerate(NET_LABS.keys()):

        anova_df = int_wmh_df[int_wmh_df.net == net_lab]
        anova_res, posthoc = pg_mix_anova(anova_df, col_name=cm_lab)
        # ANOVA result
        cm_anova_res_df = pd.concat(
            [
                cm_anova_res_df,
                anova_res.assign(
                    Network=NET_LABS[net_lab],
                    CM=cm_lab[:2].upper(),
                ),
            ],
            ignore_index=True,
        )
        # Post-hoc result
        posthoc_res_df = pd.concat(
            [
                posthoc_res_df,
                posthoc.assign(
                    Network=NET_LABS[net_lab],
                    CM=cm_lab[:2].upper(),
                ),
            ],
            ignore_index=True,
        )

    cm_anova_res_df["P"] = add_statistic_annotation(cm_anova_res_df["p-unc"].to_list())
    cm_anova_res_df["P_corrected"] = cm_anova_res_df.groupby("Source")["p-unc"].transform(
        lambda p: add_statistic_annotation(fdr_correction(p.tolist()))
    )

    anova_res_df = pd.concat([anova_res_df, cm_anova_res_df], ignore_index=True)

posthoc_res_df["P"] = add_statistic_annotation(posthoc_res_df["pval"].to_list())
anova_res_df.round(3).to_csv(f"{output_dir}/local_CFC_comparison-Dominance-INT_WMH.csv")
posthoc_res_df.round(3).to_csv(
    f"{output_dir}/local_CFC_comparison-Dominance-INT_WMH-posthoc.csv"
)

## Figure 3B&C. Swarmplot for local CFC R2

In [ ]:
comp_res = []
# Plot LR rsq
for lm_idx, tract_lab in enumerate(TRACT_LABS):
    fig, axes = plt.subplots(nrows=1, ncols=len(NET_LABS), figsize=(6, 3))
    plt.subplots_adjust(wspace=0.02)
    axes = axes.flatten()
    for net_idx, net_lab in enumerate(NET_LABS):
        # plot
        ax = axes[net_idx]
        plot_df = merged_df[
            (merged_df.Tractography == tract_lab) & (merged_df.net == net_lab)
        ]
        ylim = 0.5
        ax = plot_ax_swarmplot(plot_df, ax, net_idx, x="Group", y="rsq", swarmsize=3)
        
        ax.set_ylabel(["All", "INT", "WMH"][lm_idx])

        ax.set_title(NET_LABS[net_lab])

        ax.set_yticks(np.round(np.linspace(0, ylim, 5), 2))
        if net_idx > 0:
            ax.spines["left"].set_color("None")
            ax.set(yticks=[], ylabel="")
        if ax.get_ylim()[0] > ax.get_ylim()[1]:
            ax.invert_yaxis()
        sns.despine(trim=True, ax=ax)

    plt.tight_layout()
    plt.savefig(f"{output_dir}/local_CFC_comparison-R2-{tract_lab}.svg", bbox_inches="tight")


In [ ]:
# comp_res = []
# # Plot LR rsq
# for net_idx, net_lab in enumerate(NET_LABS):
#     fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(2, 3))
#     plt.subplots_adjust(wspace=0.02)
#     axes = axes.flatten()
#     for lm_idx, tract_lab in enumerate(TRACT_LABS[1:]):
#         # plot
#         ax = axes[lm_idx]
#         plot_df = merged_df[
#             (merged_df.Tractography == tract_lab) & (merged_df.net == net_lab)
#         ]
#         ylim = plot_df["rsq"].max() + 0.1
#         ax = plot_ax_swarmplot(plot_df, ax, net_idx, x="Group", y="rsq", swarmsize=3)
        
#         # ax.set_ylabel(["INT", "WMH"][lm_idx])

#         ax.set_yticks(np.round(np.linspace(0, ylim, 5), 2))
#         if lm_idx > 0:
#             ax.spines["left"].set_color("None")
#             ax.set(yticks=[], ylabel="")
#         if ax.get_ylim()[0] > ax.get_ylim()[1]:
#             ax.invert_yaxis()
#         sns.despine(trim=True, ax=ax)
#         ax.set_title(tract_lab.upper())
#     # plt.title(NET_LABS[net_lab])
#     plt.tight_layout()
#     plt.savefig(f"{output_dir}/local_CFC_comparison-R2-{net_lab}.svg", bbox_inches="tight")


## Figure 3E. The radar plot of CM's dominance

In [ ]:
rada_fig_size = (3, 3)
# ylim = 0.04
for net_idx, net_lab in enumerate(NET_LABS.keys()):

    # Whole brain
    tract_lab = TRACT_LABS[0]
    plot_df = merged_df[(merged_df.Tractography == tract_lab) & (merged_df.net == net_lab)]
    plot_df = plot_df[["pid", "Group"] + CM_LABS]
    plot_mean = plot_df.set_index("pid").groupby("Group").mean()
    ylim = plot_mean.values.max() + 0.002
    categories = list(plot_mean.columns)
    N = len(categories)
    fig, ax = plt.subplots(figsize=rada_fig_size, subplot_kw={"polar": True})
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    for idx, dg in enumerate(plot_mean.index):
        values = plot_mean.loc[dg].values
        values = np.concatenate((values, [values[0]]))

        ax.plot(
            angles,
            values,
            linewidth=1,
            linestyle="solid",
            color=colors_hex[DISGROUPS.index(dg)],
            label=dg,
        )
        ax.set_facecolor("white")
    pvals_list = []
    ax.set_yticks(np.round(np.linspace(0, ylim, 3), 3))
    ax.set_title(NET_LABS[net_lab])
    plt.xticks(angles[:-1], CM_ARR_SHORT)
    locs, labels = plt.xticks()
    [plt.setp(labels[i], color=cm_arr_hex[i]) for i in range(len(labels))]
    plt.savefig(
        f"{output_dir}/rada-tract_{tract_lab}-net_{net_lab}.svg",
        bbox_inches="tight",
        dpi=300,
    )

    # ------------------------
    # INT AND WMH
    # INT
    fig, ax = plt.subplots(figsize=rada_fig_size, subplot_kw={"polar": True})

    tract_lab = TRACT_LABS[1]
    plot_df = merged_df[(merged_df.Tractography == tract_lab) & (merged_df.net == net_lab)]
    plot_df = plot_df[["pid", "Group"] + CM_LABS]
    lr_rsq_mean = plot_df.set_index("pid").groupby("Group").mean()
    categories = list(lr_rsq_mean.columns)
    N = len(categories)

    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    for idx, dg in enumerate(lr_rsq_mean.index):
        values = lr_rsq_mean.loc[dg].values
        values = np.concatenate((values, [values[0]]))

        ax.plot(
            angles,
            values,
            linewidth=1,
            linestyle="solid",
            color=colors_hex[DISGROUPS.index(dg)],
            label=dg,
        )
    # --------------------------
    # WMH
    tract_lab = TRACT_LABS[2]
    plot_df = merged_df[(merged_df.Tractography == tract_lab) & (merged_df.net == net_lab)]
    plot_df = plot_df[["pid", "Group"] + CM_LABS]
    lr_rsq_mean = plot_df.set_index("pid").groupby("Group").mean()
    categories = list(lr_rsq_mean.columns)
    N = len(categories)

    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    for idx, dg in enumerate(lr_rsq_mean.index):
        values = lr_rsq_mean.loc[dg].values
        values = np.concatenate((values, [values[0]]))

        ax.plot(
            angles,
            values,
            linewidth=1,
            linestyle=":",
            color=colors_hex[DISGROUPS.index(dg)],
            label=dg,
        )
        ax.set_facecolor("white")
        ax.set_title("")

        ax.set_yticks(np.round(np.linspace(0, ylim, 3), 3))
    ax.set_title(NET_LABS[net_lab])
    plt.xticks(angles[:-1], CM_ARR_SHORT)
    locs, labels = plt.xticks()
    [plt.setp(labels[i], color=cm_arr_hex[i]) for i in range(len(labels))]
    plt.savefig(
        f"{output_dir}/rada-tract_intwmh-net_{net_lab}.svg", bbox_inches="tight", dpi=300
    )

In [ ]:
from svgutils.compose import Figure, SVG, Panel

def combine_svgs(svg_files, output_path, figure_width="1400px", figure_height="220px", spacing=200):
    """
    Combine multiple SVG files into a single figure arranged horizontally.

    Parameters:
    - svg_files: List of paths to SVG files.
    - output_path: Path to save the combined SVG figure.
    - figure_width: Width of the combined figure.
    - figure_height: Height of the combined figure.
    - spacing: Horizontal spacing between SVG panels.
    """
    # Load SVG files and create panels
    figures = [SVG(svg_file) for svg_file in svg_files]
    panels = [Panel(figure).move(i * spacing, 0) for i, figure in enumerate(figures)]
    
    # Create and save the combined figure
    combined_figure = Figure(figure_width, figure_height, *panels)
    combined_figure.save(output_path)

# Combine SVGs for whole brain (wb)
wb_svg_files = [
    f"{output_dir}/rada-tract_{TRACT_LABS[0]}-net_{net_lab}.svg"
    for net_lab in NET_LABS.keys()
]
combine_svgs(
    svg_files=wb_svg_files,
    output_path=f"{output_dir}/local_CFC_comparison-Dominance-wb.svg"
)

# Combine SVGs for INT and WMH
intwmh_svg_files = [
    f"{output_dir}/rada-tract_intwmh-net_{net_lab}.svg"
    for net_lab in NET_LABS.keys()
]
combine_svgs(
    svg_files=intwmh_svg_files,
    output_path=f"{output_dir}/local_CFC_comparison-Dominance-intwmh.svg"
)


## Figure 3F. The interaction effect.

In [ ]:
boxline_size = (2, 3.5)
cm_plot = "pl-wei"
y_lim_upper = 0.04
net_lab = "Vis"
# Iterate over each network

plot_df = merged_df[
    (merged_df.net == net_lab) & (merged_df.Tractography != TRACT_LABS[0])
]
fig, ax = plt.subplots(figsize=boxline_size)  # Create a figure and axis

sns.boxplot(
    data=plot_df,
    x="Tractography",
    y=cm_plot,
    showfliers=False,
    hue="Group",
    palette=PAL4,
    dodge=True,
    linewidth=1.5,
    linecolor="lightgray",
    hue_order=DISGROUPS,
    saturation=1,
    ax=ax,  # Pass the axis to sns.boxplot
    boxprops=dict(alpha=0.3),  
    zorder=1
)

# Calculate mean values for each group and tractography
mean_values = (
    plot_df.groupby(["Tractography", "Group"])[cm_plot].mean().unstack()
)

# Draw lines connecting the mean values within each group
for group in mean_values.columns:
    plt.plot(
        mean_values.index,
        mean_values[group],
        marker="o",
        label=f"{group} (mean)",
        linestyle="-",
        color=colors_hex[DISGROUPS.index(group)],
        zorder = 2
    )
ax.set_ylim(0, y_lim_upper)  
ax.set_xlabel("")
ax.set_ylabel("")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticklabels(["INT", "WMH"], rotation=90)
ax.set_yticks(np.round(np.linspace(0, y_lim_upper, 5), 3))
ax.set_facecolor("whitesmoke")
ax.spines["left"].set_position(("outward", 5))  # Offset the left axis
ax.spines["bottom"].set_position(("outward", 5))  # Offset the bottom axis
ax.legend_.remove()  # Remove legend to avoid overlap
plt.tight_layout()  # Adjust layout to prevent overlap

plt.savefig(
    f"{output_dir}/boxline-tract_intwmh-net_{net_lab}-cm_{cm_plot}.svg",
    bbox_inches="tight",
)
plt.show()  # Display the plot

In [ ]:
cm_lab = "pt-wei"
net_lab = "SomMot"
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(2, 3))
plt.subplots_adjust(wspace=0.02)
axes = axes.flatten()
for lm_idx, tract_lab in enumerate(TRACT_LABS[1:]):
    # plot
    ax = axes[lm_idx]
    plot_df = merged_df[
        (merged_df.Tractography == tract_lab) & (merged_df.net == net_lab)
    ]
    ylim = plot_df[cm_lab].max() + 0.03
    ax = plot_ax_swarmplot(
        plot_df, ax, net_idx, x="Group", y=cm_lab, ylim=ylim,swarmsize=3
    )

    ax.set_yticks(np.round(np.linspace(0, ylim, 5), 2))
    if lm_idx > 0:
        ax.spines["left"].set_color("None")
        ax.set(yticks=[], ylabel="")
    if ax.get_ylim()[0] > ax.get_ylim()[1]:
        ax.invert_yaxis()
    sns.despine(trim=True, ax=ax)
    ax.set_title(tract_lab.upper())
# plt.title(NET_LABS[net_lab])
plt.tight_layout()
plt.savefig(f"{output_dir}/{net_lab}_{cm_lab}.svg", bbox_inches="tight")